In [0]:
import mlflow
from mlflow.tracking import MlflowClient

catalog = "workspace"
schema = "financial_sentiment"

experiment_name = "/Users/jezapataf@eafit.edu.co/financial_sentiment_mlops"

mlflow.set_registry_uri("databricks-uc")

client = MlflowClient()
experiment = mlflow.get_experiment_by_name(experiment_name)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="params.source = 'repo_original'",
    order_by=["metrics.macro_f1 DESC"]
)

display(runs[[
    "run_id",
    "params.model_key",
    "params.hf_model_name",
    "metrics.accuracy",
    "metrics.macro_f1",
    "metrics.weighted_f1"
]])

In [0]:
MIN_MACRO_F1 = 0.50

for _, row in runs.iterrows():
    run_id = row["run_id"]
    model_key = row["params.model_key"]
    macro_f1 = row["metrics.macro_f1"]

    registered_model_name = f"{catalog}.{schema}.financial_sentiment_{model_key}"

    print("Evaluando registro:")
    print("Modelo:", registered_model_name)
    print("Run:", run_id)
    print("Macro F1:", macro_f1)

    if macro_f1 >= MIN_MACRO_F1:
        result = mlflow.register_model(
            model_uri=f"runs:/{run_id}/model",
            name=registered_model_name
        )

        print("Registrado:", result.name)
        print("Versión:", result.version)
    else:
        print("No registrado por bajo macro_f1")